# 14 — Predição individual de FA futura

Demonstração com ECGs reais do conjunto de teste. O notebook não cria exemplos sintéticos. A saída é um **escore não calibrado**, não uma probabilidade clínica.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from ecg_ai.config import load_config, resolve_project_path
from ecg_ai.dataset import read_manifest
from ecg_ai.io import load_wfdb_ecg, validate_and_reorder
from ecg_ai.predict import predict_ecg
from ecg_ai.visualization import plot_12_leads

## Seleção de um ECG real do teste

O manifesto já foi congelado por paciente. O rótulo não é passado para `predict_ecg`.

In [ ]:
cfg = load_config(ROOT / 'configs' / 'default.json')
manifest = read_manifest(resolve_project_path(cfg['data']['manifest_path']))
test = manifest.query("split == 'test'").reset_index(drop=True)
assert len(test), 'Conjunto de teste vazio'
row = test.iloc[0]
record = validate_and_reorder(load_wfdb_ecg(row.ecg_path), minimum_seconds=10.0)
plot_12_leads(record)

## Inferência individual e revelação posterior do desfecho

In [ ]:
result = predict_ecg(str(row.ecg_path))
display(result)
print('Desfecho documentado (revelado após inferência):', int(row.future_af))

## Exemplos por categoria, somente quando realmente existentes

A célula abaixo lê apenas as previsões do teste. Categorias ausentes são reportadas como ausentes; nada é fabricado.

In [ ]:
pred_path = resolve_project_path(cfg['artifacts']['predictions_path'])
pred = pd.read_parquet(pred_path)
pred['category'] = pred.apply(lambda r: ('TP' if r.y_pred else 'FN') if r.y_true else ('FP' if r.y_pred else 'TN'), axis=1)
examples = {}
for category in ['TP', 'TN', 'FP', 'FN']:
    matches = pred.query('category == @category')
    if matches.empty:
        print(f'{category}: categoria ausente nos resultados reais')
    else:
        examples[category] = matches.iloc[0].to_dict()
pd.DataFrame(examples).T if examples else pd.DataFrame()

In [ ]:
# Visualize cada exemplo real disponível e confirme o vínculo com o manifesto.
for category, example in examples.items():
    linked = test.loc[test.ecg_id.astype(str).eq(str(example['ecg_id']))]
    if linked.empty:
        print(category, 'sem vínculo no manifesto — revisar rastreabilidade')
        continue
    rec = validate_and_reorder(load_wfdb_ecg(linked.iloc[0].ecg_path), minimum_seconds=10.0)
    display(plot_12_leads(rec, title=f"{category} — {example['ecg_id']}"))
    print({k: example[k] for k in ['y_true', 'y_score', 'y_pred', 'threshold']})

## Limitações

Esta PoC depende da validade das datas, da anotação de ritmo sinusal e da captura do seguimento. O modelo não tem validação externa ou clínica. Censura informativa, mudança de instituição e diferenças demográficas podem distorcer o desempenho. O sigmoid permanece um escore não calibrado até uma avaliação de calibração prespecificada e suficientemente dimensionada. Não usar para diagnóstico ou conduta.